# P1b: frozen sealed-result analysis

Run only after notebooks 10 and 11 have completed and their reports have been reviewed. This CPU notebook integrity-checks every pre-inference decision and exhaustive loss artifact, joins frozen selections, and executes the predeclared fixed-sequence bootstrap analysis. It does not tune or revise the method.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
importlib.import_module('covsafe')
print('Git commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip())

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
PRIVATE_ROOT = Path('/content/drive/MyDrive/covariate-safe-tsfm/private_manifests')
P1B_ROOT = PRIVATE_ROOT / 'p1b'
for backbone in ('chronos_2', 'timesfm_3'):
    path = P1B_ROOT / 'reports' / f'{backbone}_p1b_completion.json'
    assert path.exists(), f'Missing completion report: {path}'
print('Both backbone completion reports found.')

In [ ]:
from covsafe.p1b import EXPECTED_P1B_CONFIG_HASH
from covsafe.p1b_analysis import run_p1b_analysis

print('Frozen P1b config hash:', EXPECTED_P1B_CONFIG_HASH)
report = run_p1b_analysis(REPO, P1B_ROOT, P1B_ROOT, P1B_ROOT)
print('Frozen analysis completed and stored durably.')

In [ ]:
import json

compact = {
    'config_hash': report['config_hash'],
    'git_commit': report['git_commit'],
    'counts': report['counts'],
    'routing': report['routing'],
    'fixed_sequence': report['fixed_sequence'],
    'per_backbone_safety': report['per_backbone_safety'],
    'worst_group_downside': report['worst_group_downside'],
    'decision': report['decision'],
}
print(json.dumps(compact, indent=2, ensure_ascii=False))

## Return artifact

Send the complete compact JSON above. Do not rerun or revise the method based on individual group results; result interpretation and claim backfill happen only after this artifact is archived.